# Etapa 4A: detectores YOLO por classe (Google Colab, GPU)

Projeto Contagem de Objetos em Imagens Industriais. Este notebook treina um detector YOLO por classe (`parafuso`, `porca`, `arruela`), calibra o limiar de confiança no valid, gera as contagens no test uma única vez e salva tudo no Google Drive. A contagem de cada imagem é o número de caixas com confiança maior ou igual ao limiar da classe. As previsões têm o mesmo formato e passam pelo mesmo avaliador (`avaliar.py`) da pipeline clássica (Etapa 3), com `metodo = "yolo26s"`.

## Antes de rodar

**1. Arquivos no Drive**, em `MyDrive/PDI_Contagem_Objetos/etapa4/`:

| Arquivo | Origem no repositório |
| --- | --- |
| `data_final.zip` | `data_final.zip` (raiz do repositório, sem alterações) |
| `avaliar.py` | cópia idêntica de `scripts/avaliar.py` |

**2. Runtime com GPU.** Use *Ambiente de execução > Alterar o tipo de ambiente de execução > GPU* (T4, L4 ou A100). A célula 1 falha com uma mensagem clara se não houver GPU.

**3. Ordem.** Execute as células de cima para baixo (*Executar tudo* funciona). Se o Colab desconectar durante o treino, reconecte e execute a célula 0 e as células 1 a 4. Depois mude `RETOMAR_TREINO = True` na célula 0 e rode a célula 5. O treino retoma do `last.pt` no Drive e as classes já concluídas são puladas.

**4. Integridade do test.** Toda decisão (treino, early stopping, escolha do limiar) usa apenas train e valid. O test só é lido na célula 8, uma única vez, depois que `config_yolo_final.json` foi congelado e seu SHA-256 registrado na célula 7. Nenhum ajuste é permitido depois de ver o resultado do test, e a célula 8 recusa uma segunda execução depois de concluída.

**5. Tempo estimado**, aproximado e dependente da GPU. As imagens de treino somam cerca de 1.050 (parafuso 510, porca 175, arruela 361). Em uma T4, as três classes levam de 40 a 90 minutos (no máximo 100 épocas, com `patience = 20`). Em L4 ou A100 costuma ser bem menos. `deterministic = True` deixa o treino mais lento. A calibração e o test levam poucos minutos.

**Saídas no Drive:** `etapa4/runs/<classe>/` (runs do Ultralytics, usados para retomar) e `etapa4/saidas/` (CSVs, config congelado, curvas, gráficos de treino, `best.pt`, imagens de exemplo e manifesto com SHA-256).


### Célula 0 — Parâmetros globais

Define a configuração única do treino no dict `CONFIG`, os valores esperados para as verificações de integridade, os caminhos e duas flags. Todo hiperparâmetro que não está em `CONFIG` fica no padrão da Ultralytics. As flags no fim da célula são as únicas opções editáveis durante a execução.


In [ ]:
from pathlib import Path

# ---------------- Configuração única (não alterar entre treino, calibração e test) ----------------
CONFIG = {
    "modelo": "yolo26s.pt",
    "imgsz": 640,
    "epochs": 100,
    "patience": 20,
    "batch": 16,
    "seed": 42,
    "deterministic": True,
    "max_det": 300,
}

METODO = "yolo26s"                      # valor da coluna `metodo` nos CSVs de previsões
CLASSES = ["parafuso", "porca", "arruela"]
CONF_MIN_INFERENCIA = 0.05              # confiança mínima da inferência única; os limiares são aplicados depois
LIMIARES = [round(0.05 * k, 2) for k in range(1, 19)]   # 0.05, 0.10, ..., 0.90

# ---------------- Verificações de integridade ----------------
# Hash do data_final.zip. O relatório da Fase 1 registrou o prefixo d542f8c06ea9bff1...;
# o valor completo abaixo é o SHA-256 do data_final.zip gerado na Fase 1 (compatível com o prefixo).
DATA_FINAL_ZIP_SHA256 = "d542f8c06ea9bff1afe9ed25debaea46f11aeddee6661a9a8fabadcda77efc81"
# SHA-256 do scripts/avaliar.py validado na Fase 2 (a cópia no Drive deve ser idêntica).
AVALIAR_SHA256 = "61c65fb0d65b751b429b7203b64651b4b1f9b5c2fcf17c1e6979fe75986a8da5"

# Conferências do GT e do conteúdo do zip (valores da Fase 1)
GT_LINHAS_ESPERADAS = 1280
GT_OBJETOS_ESPERADOS = 10980
IMAGENS_ESPERADAS = {   # imagens em data_final/yolo/<classe>/<divisão>/images
    "parafuso": {"train": 510, "valid": 40, "test": 40},
    "porca":    {"train": 175, "valid": 15, "test": 15},
    "arruela":  {"train": 361, "valid": 62, "test": 62},
}

# ---------------- Caminhos ----------------
DRIVE_RAIZ = Path("/content/drive/MyDrive/PDI_Contagem_Objetos/etapa4")
RUNS_DIR = DRIVE_RAIZ / "runs"          # runs/<classe>/ (persistem se o Colab desconectar)
SAIDAS_DIR = DRIVE_RAIZ / "saidas"      # entregáveis finais
LOCAL = Path("/content")
ZIP_LOCAL = LOCAL / "data_final.zip"
DATA_FINAL = LOCAL / "data_final"
YAMLS_DIR = LOCAL / "yamls_yolo"

# ---------------- Flags (únicas opções editáveis) ----------------
RETOMAR_TREINO = False        # True: retoma treinos interrompidos a partir do last.pt do Drive
HASH_CONFIG_ESPERADO = None   # opcional: cole aqui o SHA-256 impresso na célula 7 para conferir de novo na célula 8

### Célula 1 — Drive, GPU e versões

Monta o Drive e para com uma mensagem clara se não houver GPU. Instala o `ultralytics` e grava as versões de `ultralytics`, `torch` e CUDA em `saidas/versoes_ambiente.json`, para o relatório poder citar o ambiente.


In [ ]:
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone

from google.colab import drive
drive.mount("/content/drive")

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU NAO DISPONIVEL. No Colab use: Ambiente de execucao > Alterar o tipo de ambiente de execucao > "
        "selecione uma GPU (T4, L4 ou A100) e execute novamente desde a primeira celula. "
        "O treino em CPU nao e suportado neste notebook."
    )

for _arq in ("data_final.zip", "avaliar.py"):
    if not (DRIVE_RAIZ / _arq).is_file():
        raise FileNotFoundError(
            f"Arquivo ausente no Drive: {DRIVE_RAIZ / _arq}. Suba data_final.zip e avaliar.py "
            f"(copia de scripts/avaliar.py) para {DRIVE_RAIZ} e execute novamente."
        )
RUNS_DIR.mkdir(parents=True, exist_ok=True)
SAIDAS_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"], check=True)
import ultralytics

try:
    _driver = subprocess.run(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip().splitlines()[0]
except Exception:
    _driver = None

VERSOES = {
    "registrado_em_utc": datetime.now(timezone.utc).isoformat(),
    "ultralytics": ultralytics.__version__,
    "torch": torch.__version__,
    "cuda_torch": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "driver_nvidia": _driver,
    "python": platform.python_version(),
    "plataforma": platform.platform(),
}
(SAIDAS_DIR / "versoes_ambiente.json").write_text(json.dumps(VERSOES, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(VERSOES, indent=2, ensure_ascii=False))

### Célula 2 — Dados locais, integridade e avaliador

Copia `data_final.zip` para o disco local (`/content`) e confere o SHA-256. Se divergir, o notebook para antes de descompactar. Depois descompacta, confere o GT, copia `avaliar.py`, confere o hash dele e o importa como `av`.


In [ ]:
import hashlib
import shutil
import zipfile


def sha256_arquivo(caminho, bloco=1 << 20):
    """SHA-256 de um arquivo, lido em blocos de 1 MiB."""
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for parte in iter(lambda: f.read(bloco), b""):
            h.update(parte)
    return h.hexdigest()


shutil.copy2(DRIVE_RAIZ / "data_final.zip", ZIP_LOCAL)
sha_zip = sha256_arquivo(ZIP_LOCAL)
if sha_zip != DATA_FINAL_ZIP_SHA256:
    raise RuntimeError(
        "SHA-256 de data_final.zip DIVERGE do valor esperado (Fase 1). PARANDO.\n"
        f"  esperado: {DATA_FINAL_ZIP_SHA256}\n  obtido  : {sha_zip}"
    )
print("SHA-256 de data_final.zip conferido:", sha_zip)

if DATA_FINAL.exists():
    shutil.rmtree(DATA_FINAL)
with zipfile.ZipFile(ZIP_LOCAL) as z:
    z.extractall(LOCAL)
if not (DATA_FINAL / "yolo").is_dir() or not (DATA_FINAL / "contagens_gt.csv").is_file():
    raise RuntimeError(f"Estrutura inesperada apos descompactar: {DATA_FINAL}")

# Avaliador (cópia idêntica de scripts/avaliar.py)
shutil.copy2(DRIVE_RAIZ / "avaliar.py", LOCAL / "avaliar.py")
sha_av = sha256_arquivo(LOCAL / "avaliar.py")
if sha_av != AVALIAR_SHA256:
    raise RuntimeError(
        "avaliar.py no Drive DIVERGE do scripts/avaliar.py validado na Fase 2. PARANDO.\n"
        f"  esperado: {AVALIAR_SHA256}\n  obtido  : {sha_av}"
    )
if str(LOCAL) not in sys.path:
    sys.path.insert(0, str(LOCAL))
import avaliar as av

df_gt = av.carregar_gt(DATA_FINAL / "contagens_gt.csv")
if len(df_gt) != GT_LINHAS_ESPERADAS or int(df_gt["contagem"].sum()) != GT_OBJETOS_ESPERADOS:
    raise RuntimeError(
        f"GT inesperado: {len(df_gt)} linhas / {int(df_gt['contagem'].sum())} objetos "
        f"(esperado {GT_LINHAS_ESPERADAS} / {GT_OBJETOS_ESPERADOS})."
    )
print("avaliar.py conferido:", sha_av)
print("GT:", len(df_gt), "linhas |", int(df_gt["contagem"].sum()), "objetos |", df_gt.groupby("split").size().to_dict())

### Célula 3 — `data.yaml` com caminho absoluto

Gera, para cada classe, uma cópia de `data_final/yolo/<classe>/data.yaml` em `/content/yamls_yolo/` com `path` absoluto. Os originais não mudam. Confere também o número de imagens por divisão. O treino usa apenas train e valid.


In [ ]:
import yaml

YAMLS_DIR.mkdir(exist_ok=True)
YAML_POR_CLASSE = {}
for classe in CLASSES:
    raiz_classe = (DATA_FINAL / "yolo" / classe).resolve()
    cfg_yaml = yaml.safe_load((raiz_classe / "data.yaml").read_text(encoding="utf-8"))
    nomes = list(cfg_yaml["names"].values()) if isinstance(cfg_yaml["names"], dict) else list(cfg_yaml["names"])
    if cfg_yaml["nc"] != 1 or nomes != [classe]:
        raise RuntimeError(f"data.yaml inesperado para {classe}: nc={cfg_yaml['nc']}, names={nomes}")
    cfg_yaml["path"] = str(raiz_classe)
    destino = YAMLS_DIR / f"{classe}.yaml"
    destino.write_text(yaml.safe_dump(cfg_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8")
    YAML_POR_CLASSE[classe] = destino
    for split, n_esp in IMAGENS_ESPERADAS[classe].items():
        n = len(list((raiz_classe / split / "images").iterdir()))
        if n != n_esp:
            raise RuntimeError(f"{classe}/{split}: {n} imagens (esperado {n_esp}).")
    print(f"{classe}: {destino}  ->  path={cfg_yaml['path']}")

### Célula 4 — Configuração e verificação dos pesos

Mostra o `CONFIG` e confere que os pesos `yolo26s.pt` carregam. Se não carregarem, o notebook para com uma mensagem clara e não troca de modelo por conta própria. Pode ser preciso uma versão mais recente do `ultralytics`.


In [ ]:
from ultralytics import YOLO

print("CONFIG:", json.dumps(CONFIG, indent=2))
try:
    _modelo_teste = YOLO(CONFIG["modelo"])
except Exception as erro:
    raise RuntimeError(
        f"NAO FOI POSSIVEL CARREGAR OS PESOS '{CONFIG['modelo']}'. PARANDO (nenhum modelo alternativo sera usado). "
        f"Verifique conexao/versao do ultralytics ({VERSOES['ultralytics']}). Erro original: {erro!r}"
    ) from erro
if _modelo_teste.task != "detect":
    raise RuntimeError(f"Os pesos {CONFIG['modelo']} nao sao de deteccao (task={_modelo_teste.task}).")

_pesos = getattr(_modelo_teste, "ckpt_path", None)
_pesos = Path(_pesos) if _pesos else Path(CONFIG["modelo"])
MODELO_SHA256 = sha256_arquivo(_pesos) if _pesos.is_file() else None
print(f"Pesos carregados: {CONFIG['modelo']} | task={_modelo_teste.task} | arquivo={_pesos} | sha256={MODELO_SHA256}")
del _modelo_teste

### Célula 5 — Treino (um modelo por classe)

Treina `parafuso`, `porca` e `arruela`, cada uma em `etapa4/runs/<classe>/` no Drive, para não perder o progresso. Os hiperparâmetros são os do `CONFIG` e o restante fica no padrão da Ultralytics. O early stopping (`patience = 20`) usa o valid.

- Classe com `train_concluido.json`: é pulada.
- `RETOMAR_TREINO = True`: retoma do `last.pt` (`resume=True`).
- Pasta de run existente, não concluída e sem `RETOMAR_TREINO`: dá erro, para não sobrescrever progresso sem querer.


In [ ]:
import time


def caminhos_run(classe):
    """Devolve (pasta do run, best.pt, last.pt, train_concluido.json) da classe."""
    run = RUNS_DIR / classe
    return run, run / "weights" / "best.pt", run / "weights" / "last.pt", run / "train_concluido.json"


def treinar_classe(classe):
    """Treina (ou retoma) o detector da classe e grava train_concluido.json no fim; pula se já houver."""
    run, best, last, marcador = caminhos_run(classe)
    if marcador.is_file():
        print(f"[{classe}] treino ja concluido ({marcador.name}); pulando.")
        return
    t0 = time.time()
    if RETOMAR_TREINO and last.is_file():
        print(f"[{classe}] RETOMANDO a partir de {last}")
        YOLO(str(last)).train(resume=True)
    else:
        if run.exists() and any(run.iterdir()):
            raise RuntimeError(
                f"[{classe}] {run} ja existe e nao esta concluido. Defina RETOMAR_TREINO = True para retomar, "
                "ou apague a pasta manualmente se quiser recomecar."
            )
        print(f"[{classe}] iniciando treino ({CONFIG['modelo']})")
        YOLO(CONFIG["modelo"]).train(
            data=str(YAML_POR_CLASSE[classe]),
            imgsz=CONFIG["imgsz"],
            epochs=CONFIG["epochs"],
            patience=CONFIG["patience"],
            batch=CONFIG["batch"],
            seed=CONFIG["seed"],
            deterministic=CONFIG["deterministic"],
            max_det=CONFIG["max_det"],
            project=str(RUNS_DIR),
            name=classe,
            exist_ok=True,
            device=0,
        )
    if not best.is_file():
        raise RuntimeError(f"[{classe}] best.pt nao encontrado em {best} apos o treino.")
    marcador.write_text(
        json.dumps({"classe": classe, "concluido_em_utc": datetime.now(timezone.utc).isoformat(),
                    "segundos_desta_execucao": round(time.time() - t0, 1)}, indent=2),
        encoding="utf-8",
    )
    print(f"[{classe}] concluido em {(time.time() - t0) / 60:.1f} min (esta execucao). best.pt: {best}")


for classe in CLASSES:
    treinar_classe(classe)

### Célula 6a — Funções da calibração

Define as funções que contam as caixas acima de um limiar, fazem a inferência única no valid, montam as previsões no formato do avaliador e escolhem o limiar. O critério é o menor MAE no `avaliar.py`. Em caso de empate vale o menor |viés| e, depois, o menor limiar.


In [ ]:
import numpy as np
import pandas as pd

COLUNAS_PREVISOES = ["metodo", "dataset", "split", "imagem", "classe", "contagem"]


def mascara_acima(confs, limiar):
    """Caixas com confiança >= limiar (a tolerância de 1e-9 serve só contra ruído de ponto flutuante)."""
    return np.asarray(confs, dtype=np.float64) >= (float(limiar) - 1e-9)


def contar_acima(confs, limiar):
    """Contagem = número de caixas com confiança >= limiar."""
    return int(mascara_acima(confs, limiar).sum())


def linhas_gt_da_classe(df_gt, classe, split):
    """Linhas do GT de uma classe e divisão, em ordem estável (dataset, imagem)."""
    sub = df_gt[(df_gt["classe"] == classe) & (df_gt["split"] == split)]
    return sub.sort_values(["dataset", "imagem"]).reset_index(drop=True)


def inferir_deteccoes(model, classe, split, gt_cls, raiz_yolo):
    """Uma única inferência com confiança baixa; devolve {imagem: {dataset, conf, xyxy}} para as imagens do GT da classe."""
    nomes = list(gt_cls["imagem"])
    pasta = Path(raiz_yolo) / classe / split / "images"
    caminhos = [str(pasta / n) for n in nomes]
    ausentes = [c for c in caminhos if not Path(c).is_file()]
    if ausentes:
        raise FileNotFoundError(f"{len(ausentes)} imagem(ns) do GT ausentes em {pasta}: {ausentes[:3]}")
    dataset_de = dict(zip(gt_cls["imagem"], gt_cls["dataset"]))
    det = {}
    for r in model.predict(
        source=caminhos,
        conf=CONF_MIN_INFERENCIA,
        imgsz=CONFIG["imgsz"],
        max_det=CONFIG["max_det"],
        batch=CONFIG["batch"],
        device=0,
        half=False,
        verbose=False,
        stream=True,
    ):
        nome = Path(r.path).name
        det[nome] = {
            "dataset": dataset_de[nome],
            "conf": [float(x) for x in r.boxes.conf.cpu().numpy().tolist()],
            "xyxy": [[float(v) for v in b] for b in r.boxes.xyxy.cpu().numpy().tolist()],
        }
    if set(det) != set(nomes):
        raise RuntimeError(f"Inferencia incompleta em {classe}/{split}: {len(det)} de {len(nomes)} imagens.")
    return det


def salvar_deteccoes(det, caminho):
    """Grava as detecções (confiança e caixa por imagem) em JSON."""
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    Path(caminho).write_text(json.dumps(det), encoding="utf-8")


def carregar_deteccoes(caminho):
    """Lê o JSON gravado por salvar_deteccoes."""
    return json.loads(Path(caminho).read_text(encoding="utf-8"))


def montar_previsoes(gt_cls, det, limiar, metodo):
    """Linhas no formato do avaliador, exatamente uma por linha do GT da classe."""
    linhas = []
    for r in gt_cls.itertuples(index=False):
        linhas.append(
            {
                "metodo": metodo,
                "dataset": r.dataset,
                "split": r.split,
                "imagem": r.imagem,
                "classe": r.classe,
                "contagem": contar_acima(det[r.imagem]["conf"], limiar),
            }
        )
    return pd.DataFrame(linhas, columns=COLUNAS_PREVISOES)


def calibrar_limiar(gt_cls, det, classe, pasta_tmp):
    """Varre os limiares e avalia cada um com avaliar.py (split=valid). Devolve (curva, limiar escolhido)."""
    pasta_tmp = Path(pasta_tmp)
    pasta_tmp.mkdir(parents=True, exist_ok=True)
    caminho_gt = pasta_tmp / f"gt_valid_{classe}.csv"
    gt_cls.to_csv(caminho_gt, index=False, encoding="utf-8")
    linhas = []
    for lim in LIMIARES:
        prev = montar_previsoes(gt_cls, det, lim, METODO)
        caminho_prev = pasta_tmp / f"prev_valid_{classe}_{lim:.2f}.csv"
        prev.to_csv(caminho_prev, index=False, encoding="utf-8")
        metricas, _, _ = av.avaliar(
            caminho_previsoes=caminho_prev,
            caminho_gt=caminho_gt,
            nome_execucao=f"calib_{classe}_{lim:.2f}",
            split="valid",
            pasta_resultados=pasta_tmp / "avaliacao",
            imprimir=False,
        )
        m = metricas[(metricas["escopo"] == "classe") & (metricas["classe"] == classe)].set_index("metrica")["valor"]
        linhas.append(
            {
                "classe": classe,
                "limiar": lim,
                "mae": float(m["mae"]),
                "rmse": float(m["rmse"]),
                "vies": float(m["vies"]),
                "acerto_exato_pct": float(m["acerto_exato_pct"]),
                "acerto_tol1_pct": float(m["acerto_tol1_pct"]),
                "soma_gt": int(m["soma_gt"]),
                "soma_prevista": int(m["soma_prevista"]),
            }
        )
    curva = pd.DataFrame(linhas)
    ordenada = curva.assign(_mae=curva["mae"].round(10), _abs_vies=curva["vies"].abs().round(10)).sort_values(
        ["_mae", "_abs_vies", "limiar"]
    )
    return curva, float(ordenada.iloc[0]["limiar"])

### Célula 6b — Calibração do limiar de confiança no valid

Faz uma única inferência no valid por classe, com `conf = 0,05`, e guarda as confianças e as caixas de cada imagem. Para cada limiar de 0,05 a 0,90 (passo 0,05), a contagem é o número de caixas com confiança maior ou igual ao limiar. O MAE de cada limiar vem do `avaliar.py` (`split = valid`), só com as linhas do `contagens_gt.csv` daquela classe. A curva de limiar por MAE é salva em CSV e PNG.


In [ ]:
import matplotlib.pyplot as plt

CALIB_DIR = SAIDAS_DIR / "calibracao"
DET_DIR = SAIDAS_DIR / "deteccoes"
CALIB_DIR.mkdir(parents=True, exist_ok=True)
DET_DIR.mkdir(parents=True, exist_ok=True)

curvas, escolhidos = [], {}
for classe in CLASSES:
    _, best, _, _ = caminhos_run(classe)
    gt_val = linhas_gt_da_classe(df_gt, classe, "valid")
    det_val = inferir_deteccoes(YOLO(str(best)), classe, "valid", gt_val, DATA_FINAL / "yolo")
    salvar_deteccoes(det_val, DET_DIR / f"deteccoes_valid_{classe}.json")
    curva, limiar = calibrar_limiar(gt_val, det_val, classe, LOCAL / "tmp_calibracao")
    curva.to_csv(CALIB_DIR / f"curva_calibracao_{classe}.csv", index=False, encoding="utf-8")
    curvas.append(curva)
    mae_escolhido = float(curva.loc[curva["limiar"] == limiar, "mae"].iloc[0])
    escolhidos[classe] = {"limiar_conf": limiar, "mae_valid": mae_escolhido}
    print(f"[{classe}] limiar escolhido = {limiar:.2f} | MAE valid = {mae_escolhido:.3f}")

curva_total = pd.concat(curvas, ignore_index=True)
curva_total.to_csv(CALIB_DIR / "curva_calibracao.csv", index=False, encoding="utf-8")
(CALIB_DIR / "limiares_escolhidos.json").write_text(json.dumps(escolhidos, indent=2), encoding="utf-8")

fig, eixos = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 3.8), sharey=False)
for ax, classe in zip(np.atleast_1d(eixos), CLASSES):
    c = curva_total[curva_total["classe"] == classe]
    ax.plot(c["limiar"], c["mae"], marker="o")
    ax.axvline(escolhidos[classe]["limiar_conf"], color="red", linestyle="--", label=f"escolhido = {escolhidos[classe]['limiar_conf']:.2f}")
    ax.set_title(f"{classe} (valid)")
    ax.set_xlabel("limiar de confianca")
    ax.set_ylabel("MAE")
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(CALIB_DIR / "curva_calibracao.png", dpi=150)
plt.show()
display(curva_total.round(3))

### Célula 7 — Congelar a configuração antes de usar o test

Salva `config_yolo_final.json` (modelo, versões, caminho e SHA-256 do `best.pt` de cada classe e limiar escolhido por classe) e grava o SHA-256 do arquivo em `config_yolo_final.sha256`. Copie o hash impresso e, se quiser, cole em `HASH_CONFIG_ESPERADO`. Daqui em diante nada mais é ajustado.


In [ ]:
MARCADOR_TEST = SAIDAS_DIR / "TEST_EXECUCAO.json"
CONFIG_PATH = SAIDAS_DIR / "config_yolo_final.json"
CONFIG_SHA_PATH = SAIDAS_DIR / "config_yolo_final.sha256"

if MARCADOR_TEST.is_file() and json.loads(MARCADOR_TEST.read_text(encoding="utf-8")).get("status") == "concluido":
    raise RuntimeError("O split test ja foi executado e concluido: o config nao pode mais ser regerado.")

_limiares = json.loads((CALIB_DIR / "limiares_escolhidos.json").read_text(encoding="utf-8"))
config_final = {
    "metodo": METODO,
    "congelado_em_utc": datetime.now(timezone.utc).isoformat(),
    "nota": "Congelado apos treino e calibracao em train/valid. O split test nao foi usado ate este ponto.",
    "modelo": CONFIG["modelo"],
    "modelo_pesos_sha256": MODELO_SHA256,
    "config_treino": CONFIG,
    "calibracao": {
        "conf_minimo_inferencia": CONF_MIN_INFERENCIA,
        "grade_limiares": LIMIARES,
        "criterio": "menor MAE no valid (avaliar.py, split=valid); empate: menor |vies|, depois menor limiar",
        "regra_contagem": "numero de caixas com confianca >= limiar da classe",
    },
    "versoes": VERSOES,
    "data_final_zip_sha256": DATA_FINAL_ZIP_SHA256,
    "avaliar_py_sha256": AVALIAR_SHA256,
    "classes": {},
}
for classe in CLASSES:
    _, best, _, _ = caminhos_run(classe)
    config_final["classes"][classe] = {
        "best_pt": str(best),
        "best_pt_sha256": sha256_arquivo(best),
        "limiar_conf": _limiares[classe]["limiar_conf"],
        "mae_valid_no_limiar": _limiares[classe]["mae_valid"],
    }
CONFIG_PATH.write_text(json.dumps(config_final, indent=2, ensure_ascii=False), encoding="utf-8")
HASH_CONFIG = sha256_arquivo(CONFIG_PATH)
CONFIG_SHA_PATH.write_text(f"{HASH_CONFIG}  config_yolo_final.json\n", encoding="utf-8")

print("[CONGELADO]", CONFIG_PATH)
print("[SHA-256 config_yolo_final.json]", HASH_CONFIG)
print(json.dumps({c: v["limiar_conf"] for c, v in config_final["classes"].items()}, indent=2))

### Célula 8a — Função que gera as previsões

Define `gerar_previsoes`, que junta as previsões das três classes em uma única tabela, só com as linhas que existem no GT da divisão pedida.


In [ ]:
def gerar_previsoes(df_gt, split, deteccoes_por_classe, limiares_por_classe):
    """Concatena as previsões das três classes, só com linhas que existem no GT da divisão."""
    partes = []
    for classe in CLASSES:
        gt_c = linhas_gt_da_classe(df_gt, classe, split)
        partes.append(montar_previsoes(gt_c, deteccoes_por_classe[classe], limiares_por_classe[classe], METODO))
    df = pd.concat(partes, ignore_index=True)
    return df.sort_values(["dataset", "imagem", "classe"]).reset_index(drop=True)

### Célula 8b — Test (execução única)

> **Atenção:** esta é a única execução permitida no test. Nenhum ajuste (limiar, modelo, época, parâmetro) é permitido depois de ver estes resultados. Se algo parecer ruim, entra no relatório como resultado, e não como motivo de novo treino ou de nova calibração.

A célula (1) confere o SHA-256 do config, (2) gera as previsões do valid com os limiares congelados, (3) roda cada detector nas imagens de test dos datasets que rotulam aquela classe e conta as caixas acima do limiar da classe, (4) grava `previsoes_valid.csv` e `previsoes_test.csv` (`metodo = "yolo26s"`) e (5) roda o `avaliar.py`, cujo casamento estrito precisa passar. O marcador `TEST_EXECUCAO.json` impede uma segunda execução concluída.


In [ ]:
# --- Trava: o test só pode ser concluído uma vez ---
if MARCADOR_TEST.is_file():
    _estado = json.loads(MARCADOR_TEST.read_text(encoding="utf-8"))
    if _estado.get("status") == "concluido":
        raise RuntimeError("O split test JA foi executado e concluido. Nao execute de novo (regra: uma unica vez).")
    print("AVISO: execucao anterior iniciada e nao concluida (nenhum resultado foi registrado); repetindo com o MESMO config.")

# --- 1. Conferência do SHA-256 do config congelado ---
hash_atual = sha256_arquivo(CONFIG_PATH)
hash_registrado = CONFIG_SHA_PATH.read_text(encoding="utf-8").split()[0]
if hash_atual != hash_registrado:
    raise RuntimeError(f"SHA-256 do config DIVERGE do registrado. PARANDO.\n  registrado: {hash_registrado}\n  atual     : {hash_atual}")
if HASH_CONFIG_ESPERADO and hash_atual != HASH_CONFIG_ESPERADO:
    raise RuntimeError(f"SHA-256 do config DIVERGE de HASH_CONFIG_ESPERADO. PARANDO.\n  esperado: {HASH_CONFIG_ESPERADO}\n  atual   : {hash_atual}")
print("SHA-256 do config conferido:", hash_atual)

cfg_congelado = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
limiares_final = {c: float(cfg_congelado["classes"][c]["limiar_conf"]) for c in CLASSES}
MARCADOR_TEST.write_text(
    json.dumps({"status": "iniciado", "config_sha256": hash_atual, "iniciado_em_utc": datetime.now(timezone.utc).isoformat()}, indent=2),
    encoding="utf-8",
)

# --- 2. Previsões do valid (detecções salvas na calibração e limiares congelados) ---
det_valid = {c: carregar_deteccoes(DET_DIR / f"deteccoes_valid_{c}.json") for c in CLASSES}
prev_valid = gerar_previsoes(df_gt, "valid", det_valid, limiares_final)
caminho_prev_valid = SAIDAS_DIR / "previsoes_valid.csv"
prev_valid.to_csv(caminho_prev_valid, index=False, encoding="utf-8")

# --- 3. Inferência no test (uma única vez) ---
det_test = {}
for classe in CLASSES:
    info = cfg_congelado["classes"][classe]
    if sha256_arquivo(info["best_pt"]) != info["best_pt_sha256"]:
        raise RuntimeError(f"best.pt de {classe} mudou desde o congelamento. PARANDO.")
    gt_test = linhas_gt_da_classe(df_gt, classe, "test")
    det_test[classe] = inferir_deteccoes(YOLO(info["best_pt"]), classe, "test", gt_test, DATA_FINAL / "yolo")
    salvar_deteccoes(det_test[classe], DET_DIR / f"deteccoes_test_{classe}.json")
    print(f"[{classe}] test: {len(det_test[classe])} imagens | limiar = {limiares_final[classe]:.2f}")

prev_test = gerar_previsoes(df_gt, "test", det_test, limiares_final)
caminho_prev_test = SAIDAS_DIR / "previsoes_test.csv"
prev_test.to_csv(caminho_prev_test, index=False, encoding="utf-8")

# --- 4. Avaliação com avaliar.py (casamento estrito com o GT) ---
AV_DIR = SAIDAS_DIR / "avaliacao"
print("\n=== VALID (limiares congelados) ===")
av.avaliar(caminho_prev_valid, DATA_FINAL / "contagens_gt.csv", nome_execucao=f"{METODO}_valid", split="valid", pasta_resultados=AV_DIR)
print("\n=== TEST (resultado final) ===")
metricas_test, por_imagem_test, _ = av.avaliar(caminho_prev_test, DATA_FINAL / "contagens_gt.csv", nome_execucao=f"{METODO}_test", split="test", pasta_resultados=AV_DIR)

MARCADOR_TEST.write_text(
    json.dumps({"status": "concluido", "config_sha256": hash_atual, "concluido_em_utc": datetime.now(timezone.utc).isoformat()}, indent=2),
    encoding="utf-8",
)

### Célula 9a — Funções para as imagens de exemplo

Define `desenhar_amostra`, que desenha as caixas previstas e uma faixa com previsto e GT por classe, e `sortear_imagens`, que sorteia as imagens com semente fixa.


In [ ]:
import cv2
import random

CORES_BGR = {"parafuso": (0, 140, 255), "porca": (255, 90, 0), "arruela": (0, 190, 0)}


def desenhar_amostra(caminho_img, classes_ds, deteccoes, limiares, gt_contagens, destino):
    """Desenha as caixas com confiança >= limiar de cada classe do dataset, com a faixa de texto acima da imagem."""
    img = cv2.imread(str(caminho_img), cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"Imagem ilegivel: {caminho_img}")
    nome = Path(caminho_img).name
    previstos = {}
    for classe in classes_ds:
        d = deteccoes[classe][nome]
        confs = np.asarray(d["conf"], dtype=np.float64)
        caixas = np.asarray(d["xyxy"], dtype=np.float64).reshape(-1, 4)
        manter = mascara_acima(confs, limiares[classe]) if len(confs) else np.zeros(0, dtype=bool)
        previstos[classe] = int(manter.sum())
        for (x1, y1, x2, y2), c in zip(caixas[manter], confs[manter]):
            cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), CORES_BGR[classe], 2)
            cv2.putText(img, f"{c:.2f}", (int(x1), max(10, int(y1) - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, CORES_BGR[classe], 1, cv2.LINE_AA)
    faixa = np.full((28, img.shape[1], 3), 245, dtype=np.uint8)
    x = 6
    for classe in classes_ds:
        txt = f"{classe}: prev={previstos[classe]} gt={gt_contagens[classe]}"
        cv2.putText(faixa, txt, (x, 19), cv2.FONT_HERSHEY_SIMPLEX, 0.48, CORES_BGR[classe], 1, cv2.LINE_AA)
        x += 9 * len(txt) + 14
    saida = np.vstack([faixa, img])
    Path(destino).parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(destino), saida)
    return previstos


def sortear_imagens(df_gt, dataset, split, n=3, seed=42):
    """Sorteia, com semente fixa, até n imagens do dataset e da divisão."""
    nomes = sorted(df_gt[(df_gt["dataset"] == dataset) & (df_gt["split"] == split)]["imagem"].unique())
    return sorted(random.Random(seed).sample(nomes, min(n, len(nomes))))

### Célula 9b — Salvar os entregáveis no Drive

Em `etapa4/saidas/`, reúne as previsões (já gravadas ali), o `config_yolo_final.json` com o `.sha256`, as curvas de calibração, o `results.csv` e os gráficos de treino de cada classe (`saidas/treino/<classe>/`) e os `best.pt` (`saidas/pesos/`). Salva 3 imagens de test por dataset com as caixas previstas, sorteadas com semente 42 e sem seleção manual, e um manifesto com o SHA-256 de todos os arquivos.


In [ ]:
import shutil

# --- Imagens de test com as caixas previstas (3 por dataset) ---
AMOSTRAS_DIR = SAIDAS_DIR / "amostras_test"
classes_do_dataset = df_gt.groupby("dataset")["classe"].unique().to_dict()
for dataset, classes_ds in sorted(classes_do_dataset.items()):
    classes_ds = [c for c in CLASSES if c in set(classes_ds)]
    for nome in sortear_imagens(df_gt, dataset, "test", n=3, seed=42):
        gt_c = {
            c: int(df_gt[(df_gt["dataset"] == dataset) & (df_gt["split"] == "test") & (df_gt["imagem"] == nome) & (df_gt["classe"] == c)]["contagem"].iloc[0])
            for c in classes_ds
        }
        origem = DATA_FINAL / "base" / dataset / "test" / "images" / nome
        desenhar_amostra(origem, classes_ds, det_test, limiares_final, gt_c, AMOSTRAS_DIR / dataset / (Path(nome).stem + "_yolo.jpg"))
print("Amostras salvas em", AMOSTRAS_DIR)

# --- Artefatos de treino e pesos ---
for classe in CLASSES:
    run, best, _, _ = caminhos_run(classe)
    if not (run / "results.csv").is_file():
        raise FileNotFoundError(f"results.csv ausente em {run}")
    destino = SAIDAS_DIR / "treino" / classe
    destino.mkdir(parents=True, exist_ok=True)
    for arq in [run / "results.csv", run / "args.yaml"] + sorted(run.glob("*.png")):
        if arq.is_file():
            shutil.copy2(arq, destino / arq.name)
    (SAIDAS_DIR / "pesos").mkdir(exist_ok=True)
    shutil.copy2(best, SAIDAS_DIR / "pesos" / f"{classe}_best.pt")

# --- Manifesto com o SHA-256 de cada arquivo ---
manifesto = []
for arq in sorted(SAIDAS_DIR.rglob("*")):
    if arq.is_file() and arq.name != "manifesto_saidas.json":
        manifesto.append({"arquivo": str(arq.relative_to(SAIDAS_DIR)), "bytes": arq.stat().st_size, "sha256": sha256_arquivo(arq)})
(SAIDAS_DIR / "manifesto_saidas.json").write_text(
    json.dumps({"config_yolo_final_sha256": hash_atual, "arquivos": manifesto}, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"{len(manifesto)} arquivos em {SAIDAS_DIR}")
display(pd.DataFrame(manifesto)[["arquivo", "bytes"]])